In [3]:
# ===============================================================
# 🚀 Sentence-BERT Model for Question Answering on Water Footprint Dataset
# ===============================================================

# --- Mount Google Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Import dependencies ---
import os
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch

# --- Load your dataset from Drive ---
data_path = "/content/drive/MyDrive/WaterFootprintData/Maharashtra_WaterFootprint_AI.csv"
df = pd.read_csv(data_path)

# ✅ Check columns
print("Columns:", df.columns.tolist())
df.head()

# --- Define which columns to use for semantic search ---
# We'll combine all relevant columns into one text column
df["text"] = (
    df["State"].astype(str) + " " +
    df["District"].astype(str) + " " +
    df["Crop"].astype(str) + " " +
    df["Season"].astype(str) + " " +
    df["Irrigation_Type"].astype(str) +
    " total water footprint " + df["Total_Water_Footprint(L_per_kg)"].astype(str) +
    " category " + df["Category"].astype(str)
)

# --- Initialize Sentence-BERT model ---
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

# --- Encode dataset for semantic search ---
corpus_embeddings = model.encode(df["text"].tolist(), convert_to_tensor=True, show_progress_bar=True)

# --- Save model and embeddings to Google Drive ---
drive_folder = "/content/drive/MyDrive/Model_training"
os.makedirs(drive_folder, exist_ok=True)

model_save_path = os.path.join(drive_folder, "water_footprint_sentencebert")
model.save(model_save_path)

torch.save(corpus_embeddings, os.path.join(drive_folder, "embeddings.pt"))
df.to_csv(os.path.join(drive_folder, "water_footprint_texts.csv"), index=False)

print(f"✅ Model and embeddings saved successfully to: {drive_folder}")

# --- 🔍 Define simple question-answer function ---
def answer_query(query, top_k=3):
    query_embedding = model.encode(query, convert_to_tensor=True)
    hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=top_k)[0]

    print(f"\n🔹 User Query: {query}\n")
    for i, hit in enumerate(hits):
        row = df.iloc[hit['corpus_id']]
        print(f"Rank {i+1}:")
        print(f"  Crop: {row['Crop']}")
        print(f"  District: {row['District']}")
        print(f"  Total Water Footprint: {row['Total_Water_Footprint(L_per_kg)']} L/kg")
        print(f"  Category: {row['Category']}")
        print(f"  Score: {hit['score']:.4f}\n")

# --- Example query ---
answer_query("What is the water footprint of wheat in Nagpur?")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Columns: ['State', 'District', 'Crop', 'Season', 'Irrigation_Type', 'Total_Water_Footprint(L_per_kg)', 'Category']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

✅ Model and embeddings saved successfully to: /content/drive/MyDrive/Model_training

🔹 User Query: What is the water footprint of wheat in Nagpur?

Rank 1:
  Crop: Wheat
  District: Nagpur
  Total Water Footprint: 1607.77 L/kg
  Category: Medium
  Score: 0.7953

Rank 2:
  Crop: Wheat
  District: Sangli
  Total Water Footprint: 1451.35 L/kg
  Category: Medium
  Score: 0.7505

Rank 3:
  Crop: Wheat
  District: Pune
  Total Water Footprint: 1624.86 L/kg
  Category: Medium
  Score: 0.7496



In [4]:
# ================================================================
# 🤖 Maharashtra Water Footprint Chatbot (NLP + Sentence-BERT)
# ================================================================

# --- Step 1: Mount Google Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Step 2: Import dependencies ---
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
import torch

# --- Step 3: Define paths ---
data_path = "/content/drive/MyDrive/WaterFootprintData/maharashtra_water_footprint_ai_generated.csv"
drive_folder = "/content/drive/MyDrive/Model_training"
os.makedirs(drive_folder, exist_ok=True)

# --- Step 4: Load dataset ---
df = pd.read_csv("/content/drive/MyDrive/WaterFootprintData/maharashtra_water_footprint_ai_generated.csv")
print("✅ Dataset loaded successfully!\n")


# --- Step 5: Create text column for semantic meaning ---
df["text"] = (
    "The total water footprint of "
    + df["Crop"].astype(str)
    + " in "
    + df["District"].astype(str)
    + " during "
    + df["Season"].astype(str)
    + " season under "
    + df["Irrigation_Type"].astype(str)
    + " irrigation is "
    + df["Total_Water_Footprint(L_per_kg)"].astype(str)
    + " L/kg, which is categorized as "
    + df["Category"].astype(str)
    + "."
)

# --- Step 6: Load Sentence-BERT Model ---
model_path = os.path.join(drive_folder, "maharashtra_water_model")
if os.path.exists(model_path):
    print("📦 Loading saved model from Drive...")
    model = SentenceTransformer(model_path)
else:
    print("🧠 Training Sentence-BERT model (this may take a few minutes)...")
    model = SentenceTransformer("all-MiniLM-L6-v2")
    model.save(model_path)
    print(f"✅ Model saved to: {model_path}")

# --- Step 7: Create or load embeddings ---
embedding_path = os.path.join(drive_folder, "maharashtra_water_embeddings.pt")

if os.path.exists(embedding_path):
    print("📂 Loading pre-saved embeddings from Drive...")
    embeddings = torch.load(embedding_path)
else:
    print("⚙️ Generating embeddings (first-time only)...")
    embeddings = model.encode(df["text"].tolist(), convert_to_tensor=True, show_progress_bar=True)
    torch.save(embeddings, embedding_path)
    print(f"✅ Embeddings saved successfully to: {embedding_path}")

# --- Step 8: Define NLP search + chatbot response function ---
def chatbot_reply(user_query):
    """Finds the most relevant record and generates a chatbot-style response."""
    query_emb = model.encode(user_query, convert_to_tensor=True)
    hits = util.semantic_search(query_emb, embeddings, top_k=3)[0]

    if not hits:
        return "Sorry, I couldn’t find any relevant data for your question."

    best = hits[0]
    row = df.iloc[best["corpus_id"]]

    crop = row["Crop"]
    district = row["District"]
    value = row["Total_Water_Footprint(L_per_kg)"]
    category = row["Category"]
    irrigation = row["Irrigation_Type"]
    season = row["Season"]

    response = (
        f"Hello! 🌿 The estimated total water footprint of **{crop}** "
        f"in **{district}**, Maharashtra during the **{season}** season "
        f"under **{irrigation}** irrigation is approximately "
        f"**{value:.2f} L/kg**, which falls under the **{category.lower()}** category."
    )

    return response

# --- Step 9: Test chatbot ---
print("\n🔹 Try an example query:")
user_query = "What is the water footprint of wheat in Nagpur?"
response = chatbot_reply(user_query)
print("\n💬 Chatbot Reply:\n", response)

# --- Step 10: (Optional) You can now call chatbot_reply() dynamically ---
# e.g. chatbot_reply("Tell me about rice in Solapur")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Dataset loaded successfully!

📦 Loading saved model from Drive...
📂 Loading pre-saved embeddings from Drive...

🔹 Try an example query:

💬 Chatbot Reply:
 Hello! 🌿 The estimated total water footprint of **Wheat** in **Nagpur**, Maharashtra during the **Kharif** season under **Drip** irrigation is approximately **1502.03 L/kg**, which falls under the **medium** category.


In [2]:
# ================================================================
# 🤖 Maharashtra Water Footprint Chatbot (NLP + Sentence-BERT)
# ================================================================

# --- Step 1: Mount Google Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Step 2: Import dependencies ---
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
import torch
import re

# --- Step 3: Define paths ---
data_path = "/content/drive/MyDrive/WaterFootprintData/Maharashtra_WaterFootprint_AI.csv"
drive_folder = "/content/drive/MyDrive/Model_training"
os.makedirs(drive_folder, exist_ok=True)

# --- Step 4: Load dataset ---
df = pd.read_csv(data_path)
print("✅ Dataset loaded successfully!\n")
print("Columns:", df.columns.tolist())

# --- Step 5: Create descriptive text for each row ---
df["text"] = (
    "The total water footprint of "
    + df["Crop"].astype(str)
    + " in "
    + df["District"].astype(str)
    + " during the "
    + df["Season"].astype(str)
    + " season under "
    + df["Irrigation_Type"].astype(str)
    + " irrigation is "
    + df["Total_Water_Footprint(L_per_kg)"].astype(str)
    + " L/kg, which is categorized as "
    + df["Category"].astype(str)
    + "."
)

# --- Step 6: Load or train Sentence-BERT model ---
model_path = os.path.join(drive_folder, "maharashtra_water_model")
if os.path.exists(model_path):
    print("📦 Loading saved model from Drive...")
    model = SentenceTransformer(model_path)
else:
    print("🧠 Training Sentence-BERT model (first-time setup)...")
    model = SentenceTransformer("all-MiniLM-L6-v2")
    model.save(model_path)
    print(f"✅ Model saved to: {model_path}")

# --- Step 7: Create or load embeddings ---
embedding_path = os.path.join(drive_folder, "maharashtra_water_embeddings.pt")

if os.path.exists(embedding_path):
    print("📂 Loading pre-saved embeddings from Drive...")
    embeddings = torch.load(embedding_path)
else:
    print("⚙️ Generating embeddings (this may take a few minutes)...")
    embeddings = model.encode(df["text"].tolist(), convert_to_tensor=True, show_progress_bar=True)
    torch.save(embeddings, embedding_path)
    print(f"✅ Embeddings saved successfully to: {embedding_path}")

# --- Step 8: Define chatbot logic with NLP understanding ---
def preprocess_query(q):
    """Simple text normalization to improve matching."""
    q = q.lower().strip()
    q = re.sub(r"[?.!]", "", q)
    q = q.replace("water requirement", "water footprint")
    q = q.replace("usage", "water footprint")
    q = q.replace("consume", "water footprint")
    q = q.replace("require", "water footprint")
    return q

def chatbot_reply(user_query):
    """Enhanced function to handle natural language queries."""
    user_query = preprocess_query(user_query)
    query_emb = model.encode(user_query, convert_to_tensor=True)
    hits = util.semantic_search(query_emb, embeddings, top_k=3)[0]

    if not hits:
        return "Sorry, I couldn’t find any relevant data for your question."

    best = hits[0]
    row = df.iloc[best["corpus_id"]]

    crop = row["Crop"]
    district = row["District"]
    value = row["Total_Water_Footprint(L_per_kg)"]
    category = row["Category"]
    irrigation = row["Irrigation_Type"]
    season = row["Season"]

    # Core chatbot response
    response = (
        f"Hello! 🌿 Based on Maharashtra records, the estimated total water footprint "
        f"of **{crop}** grown in **{district}** during the **{season}** season "
        f"under **{irrigation}** irrigation is approximately **{value:.2f} L/kg**, "
        f"which falls under the **{category.lower()}** category."
    )

    # Add conversational context for certain query types
    if any(word in user_query for word in ["high", "consume", "more", "large"]):
        response += " 💧 This indicates it is relatively water-intensive compared to other crops."
    elif any(word in user_query for word in ["low", "less", "small", "efficient"]):
        response += " 🌱 This means it is relatively water-efficient."

    return response

# --- Step 9: Test chatbot with multiple natural questions ---
test_queries = [
    "What is the water footprint of wheat in Nagpur?",
    "Tell me about rice in Solapur",
    "Is cotton a high water consuming crop in Nagpur?",
    "How much water does sugarcane need in Kolhapur?",
    "Hey, can you give me the average water footprint of banana in Aurangabad?"
]

for i, q in enumerate(test_queries, start=1):
    print(f"\n🔹 Query {i}: {q}")
    print("💬 Chatbot Reply:\n", chatbot_reply(q))
    print("------------------------------------------------------")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Dataset loaded successfully!

Columns: ['State', 'District', 'Crop', 'Season', 'Irrigation_Type', 'Total_Water_Footprint(L_per_kg)', 'Category']
📦 Loading saved model from Drive...
📂 Loading pre-saved embeddings from Drive...

🔹 Query 1: What is the water footprint of wheat in Nagpur?
💬 Chatbot Reply:
 Hello! 🌿 Based on Maharashtra records, the estimated total water footprint of **Sugarcane** grown in **Pune** during the **Kharif** season under **Drip** irrigation is approximately **1870.01 L/kg**, which falls under the **medium** category.
------------------------------------------------------

🔹 Query 2: Tell me about rice in Solapur
💬 Chatbot Reply:
 Hello! 🌿 Based on Maharashtra records, the estimated total water footprint of **Maize** grown in **Nagpur** during the **Kharif** season under **Drip** irrigation is approximately **625.05 L/kg**, which fall

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [5]:
hard_queries = [
    "Which crop wastes the most water in Maharashtra?",
    "In Nashik, which crop is the most water-efficient?",
    "Does Tur require less irrigation than cotton?",
    "Between wheat and rice, which is more water-intensive?",
    "Tell me which crop saves water in Solapur district.",
]

for i, q in enumerate(hard_queries, start=1):
    print(f"\n🔹 Hard Query {i}: {q}")
    print("💬 Chatbot Reply:\n", chatbot_reply(q))
    print("------------------------------------------------------")



🔹 Hard Query 1: Which crop wastes the most water in Maharashtra?
💬 Chatbot Reply:
 Hello! 🌿 The estimated total water footprint of **Wheat** in **Pune**, Maharashtra during the **Zaid** season under **Rainfed** irrigation is approximately **1471.82 L/kg**, which falls under the **low** category.
------------------------------------------------------

🔹 Hard Query 2: In Nashik, which crop is the most water-efficient?
💬 Chatbot Reply:
 Hello! 🌿 The estimated total water footprint of **Wheat** in **Nashik**, Maharashtra during the **Zaid** season under **Well** irrigation is approximately **2092.36 L/kg**, which falls under the **medium** category.
------------------------------------------------------

🔹 Hard Query 3: Does Tur require less irrigation than cotton?
💬 Chatbot Reply:
 Hello! 🌿 The estimated total water footprint of **Tur** in **Solapur**, Maharashtra during the **Rabi** season under **Rainfed** irrigation is approximately **640.83 L/kg**, which falls under the **low** categ

In [7]:
# ================================================================
# 🤖 Smarter Query Handler for FAISS + Sentence-BERT
# ================================================================

def get_average_wf(crop=None, district=None):
    """Return average water footprint based on filters."""
    df_f = df.copy()
    if crop:
        df_f = df_f[df_f["Crop"].str.lower() == crop.lower()]
    if district:
        df_f = df_f[df_f["District"].str.lower() == district.lower()]
    if df_f.empty:
        return None
    return df_f["Total_Water_Footprint(L_per_kg)"].astype(float).mean()

def most_or_least_crop(district=None, metric="max"):
    """Return crop with highest or lowest avg footprint."""
    df_f = df.copy()
    if district:
        df_f = df_f[df_f["District"].str.lower() == district.lower()]
    grouped = df_f.groupby("Crop")["Total_Water_Footprint(L_per_kg)"].mean().reset_index()
    if grouped.empty:
        return None
    row = grouped.loc[grouped["Total_Water_Footprint(L_per_kg)"].idxmax() if metric=="max"
                      else grouped["Total_Water_Footprint(L_per_kg)"].idxmin()]
    return row["Crop"], float(row["Total_Water_Footprint(L_per_kg)"])

def answer_query(query, top_k=5):
    query_lower = query.lower()
    crop, district = extract_entities_from_query(query)

    # ---------- INTENT DETECTION ----------
    # 1️⃣ Compare crops
    if "between" in query_lower or "compare" in query_lower or "vs" in query_lower:
        crops_in_query = [c for c in valid_crops if c.lower() in query_lower]
        if len(crops_in_query) < 2:
            return "Please specify two crops to compare (e.g., 'compare wheat and rice')."
        c1, c2 = crops_in_query[:2]
        m1, m2 = get_average_wf(c1, district), get_average_wf(c2, district)
        if m1 is None or m2 is None:
            return f"No sufficient data to compare {c1} and {c2}."
        if m1 > m2:
            comp = f"**{c1}** uses more water than **{c2}**."
        elif m1 < m2:
            comp = f"**{c2}** uses more water than **{c1}**."
        else:
            comp = "Both crops have roughly equal water footprints."
        return f"📊 Comparison in {district or 'Maharashtra'}:\n- {c1}: {m1:.2f} L/kg\n- {c2}: {m2:.2f} L/kg\n{comp}"

    # 2️⃣ Most or least water-consuming crop
    if any(word in query_lower for word in ["most water", "wastes the most", "highest water", "most intensive"]):
        result = most_or_least_crop(district, metric="max")
        if result:
            crop_m, val = result
            return f"💧 The crop that wastes the most water{f' in {district}' if district else ''} is **{crop_m}**, about **{val:.2f} L/kg**."
        else:
            return "Couldn't find matching data."

    if any(word in query_lower for word in ["least water", "efficient", "saves water", "uses the least"]):
        result = most_or_least_crop(district, metric="min")
        if result:
            crop_m, val = result
            return f"🌱 The most water-efficient crop{f' in {district}' if district else ''} is **{crop_m}**, about **{val:.2f} L/kg**."
        else:
            return "Couldn't find matching data."

    # 3️⃣ Simple factual query (average for crop/district)
    if crop:
        avg = get_average_wf(crop, district)
        if avg:
            return f"📘 The average water footprint of **{crop}**{f' in {district}' if district else ''} is **{avg:.2f} L/kg**."
        else:
            return f"No records found for {crop}{' in ' + district if district else ''}."

    # 4️⃣ Fallback: Semantic similarity using FAISS
    query_embed = model.encode([query]).astype("float32")
    scores, results = index.search(query_embed, top_k)
    results = results[0]
    scores = scores[0]

    response_lines = []
    for idx, score in zip(results, scores):
        row = df.iloc[idx]
        response_lines.append(f"{row['sentence']} (score={score:.4f})")
    return "Here are the closest matches I found:\n" + "\n".join(response_lines)


In [8]:
hard_queries = [
    # Direct water footprint lookup
    "What is the water footprint of the crop sugarcane in Pune district?",

    # Efficiency-based
    "Which crop has the lowest water footprint in Nashik district?",

    # Wasteful crop identification
    "Which crop has the highest water footprint in Solapur district?",

    # Crop comparison
    "Compare the water footprint of rice and wheat crops in Satara district.",

    # Region-specific query
    "Tell me the average water footprint of soybean crop in Latur district.",

    # General question
    "In which district does the cotton crop have the highest water footprint?",

    # Water-saving crop
    "Which crop saves the most water according to the water footprint data for Ahmednagar district?",

    # Validation check (unknown entity fallback)
    "What is the water footprint of grape crop in Jalgaon district?",

    # Complex comparison
    "Between tur and groundnut crops, which has a higher water footprint in Kolhapur district?",

    # Generic overall
    "Which crop has the maximum water footprint across all districts in Maharashtra?"
]

for i, q in enumerate(hard_queries, start=1):
    print(f"\n🔹 Hard Query {i}: {q}")
    print("💬 Chatbot Reply:\n", answer_query(q))
    print("------------------------------------------------------")



🔹 Hard Query 1: What is the water footprint of the crop sugarcane in Pune district?


NameError: name 'extract_entities_from_query' is not defined

In [14]:
# ================================================================
# 🤖 Maharashtra Water Footprint Chatbot (Smart NLP + Sentence-BERT)
# ================================================================

# --- Step 1: Mount Google Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Step 2: Import dependencies ---
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
import torch
import re

# --- Step 3: Define paths ---
data_path = "/content/drive/MyDrive/WaterFootprintData/Maharashtra_WaterFootprint_AI.csv"
drive_folder = "/content/drive/MyDrive/Model_training"
os.makedirs(drive_folder, exist_ok=True)

# --- Step 4: Load dataset ---
df = pd.read_csv(data_path)
print("✅ Dataset loaded successfully!\n")
print("Columns:", df.columns.tolist())

# --- Step 5: Create descriptive text for embeddings ---
df["text"] = (
    "The total water footprint of "
    + df["Crop"].astype(str)
    + " in "
    + df["District"].astype(str)
    + " during the "
    + df["Season"].astype(str)
    + " season under "
    + df["Irrigation_Type"].astype(str)
    + " irrigation is "
    + df["Total_Water_Footprint(L_per_kg)"].astype(str)
    + " L/kg, categorized as "
    + df["Category"].astype(str)
    + "."
)

# --- Step 6: Load or train Sentence-BERT model ---
model_path = os.path.join(drive_folder, "maharashtra_water_model")
if os.path.exists(model_path):
    print("📦 Loading saved model from Drive...")
    model = SentenceTransformer(model_path)
else:
    print("🧠 Training Sentence-BERT model (first-time setup)...")
    model = SentenceTransformer("all-MiniLM-L6-v2")
    model.save(model_path)
    print(f"✅ Model saved to: {model_path}")

# --- Step 7: Create or load embeddings ---
embedding_path = os.path.join(drive_folder, "maharashtra_water_embeddings.pt")
if os.path.exists(embedding_path):
    print("📂 Loading pre-saved embeddings from Drive...")
    embeddings = torch.load(embedding_path)
else:
    print("⚙️ Generating embeddings (this may take a few minutes)...")
    embeddings = model.encode(df["text"].tolist(), convert_to_tensor=True, show_progress_bar=True)
    torch.save(embeddings, embedding_path)
    print(f"✅ Embeddings saved successfully to: {embedding_path}")

# ================================================================
# --- Step 8: Smart Chatbot Logic ---
# ================================================================

def preprocess_query(q):
    """Normalize text for better semantic matching."""
    q = q.lower().strip()
    q = re.sub(r"[?.!]", "", q)
    q = q.replace("water requirement", "water footprint")
    q = q.replace("usage", "water footprint")
    q = q.replace("consume", "water footprint")
    q = q.replace("require", "water footprint")
    return q

def detect_intent(q):
    """Identify query type (max, min, compare, lookup)."""
    if any(word in q for word in ["highest", "most", "maximum", "uses more", "largest"]):
        return "max"
    elif any(word in q for word in ["lowest", "least", "minimum", "efficient", "saves"]):
        return "min"
    elif "between" in q or "compare" in q:
        return "compare"
    else:
        return "lookup"

def extract_entities(q):
    """Extract crop and district names present in the query."""
    q = q.lower()
    crops = [c for c in df["Crop"].dropna().unique() if c.lower() in q]
    districts = [d for d in df["District"].dropna().unique() if d.lower() in q]
    return crops, districts

def chatbot_reply(user_query):
    """Core chatbot function with intelligent reasoning and fallback."""
    q = preprocess_query(user_query)
    intent = detect_intent(q)
    crops, districts = extract_entities(q)

    # --- Case 1: Specific crop + district lookup ---
    if intent == "lookup" and crops and districts:
        crop, district = crops[0], districts[0]
        subset = df[(df["Crop"] == crop) & (df["District"] == district)]
        if not subset.empty:
            row = subset.iloc[0]
            return (f"🌿 The total water footprint of **{crop}** in **{district}** "
                    f"during the **{row['Season']}** season under **{row['Irrigation_Type']}** irrigation "
                    f"is **{row['Total_Water_Footprint(L_per_kg)']:.2f} L/kg**, "
                    f"categorized as **{row['Category'].lower()}**.")

    # --- Case 2: District-wise max/min query ---
    if intent in ["max", "min"] and districts:
        district = districts[0]
        subset = df[df["District"] == district]
        if not subset.empty:
            row = subset.loc[
                subset["Total_Water_Footprint(L_per_kg)"].idxmax()
            ] if intent == "max" else subset.loc[
                subset["Total_Water_Footprint(L_per_kg)"].idxmin()
            ]
            level = "highest" if intent == "max" else "lowest"
            return (f"💧 In **{district}**, the crop with the **{level} water footprint** "
                    f"is **{row['Crop']}** with **{row['Total_Water_Footprint(L_per_kg)']:.2f} L/kg**, "
                    f"categorized as **{row['Category'].lower()}**.")

    # --- Case 3: Compare crops within same district ---
    if intent == "compare" and len(crops) == 2 and districts:
        district = districts[0]
        subset = df[(df["District"] == district) & (df["Crop"].isin(crops))]
        if len(subset) == 2:
            c1, c2 = subset.iloc[0], subset.iloc[1]
            more = c1 if c1["Total_Water_Footprint(L_per_kg)"] > c2["Total_Water_Footprint(L_per_kg)"] else c2
            less = c1 if c1["Total_Water_Footprint(L_per_kg)"] < c2["Total_Water_Footprint(L_per_kg)"] else c2
            return (f"📊 In **{district}**, **{more['Crop']}** has a higher water footprint "
                    f"({more['Total_Water_Footprint(L_per_kg)']:.2f} L/kg) than **{less['Crop']}** "
                    f"({less['Total_Water_Footprint(L_per_kg)']:.2f} L/kg).")

    # --- Case 4: Semantic similarity fallback (no direct match) ---
    query_emb = model.encode(q, convert_to_tensor=True)
    hits = util.semantic_search(query_emb, embeddings, top_k=3)[0]
    if not hits:
        return "❌ Sorry, I couldn’t find relevant data for your question."

    best = hits[0]
    row = df.iloc[best["corpus_id"]]
    response = (f"🌾 Based on available data, **{row['Crop']}** in **{row['District']}** "
                f"during the **{row['Season']}** season under **{row['Irrigation_Type']}** irrigation "
                f"has a total water footprint of **{row['Total_Water_Footprint(L_per_kg)']:.2f} L/kg**, "
                f"categorized as **{row['Category'].lower()}**.")

    if intent == "max":
        response += " 💧 This crop is considered water-intensive."
    elif intent == "min":
        response += " 🌱 This crop is relatively water-efficient."

    return response


# ================================================================
# --- Step 9: Testing with various query types ---
# ================================================================
test_queries = [
    "What is the water footprint of sugarcane in Pune district?",
    "Which crop has the highest water footprint in Solapur?",
    "Compare rice and wheat in Nagpur district.",
    "Tell me the lowest water consuming crop in Aurangabad.",
    "Give me data about cotton in Kolhapur."
]

print("\n================= 💧 Chatbot Evaluation =================\n")
for i, q in enumerate(test_queries, start=1):
    print(f"🔹 Query {i}: {q}")
    try:
        print("💬 Chatbot Reply:\n", chatbot_reply(q))
    except Exception as e:
        print(f"⚠️ Error: {e}")
    print("------------------------------------------------------")
print("\n✅ All queries processed successfully!\n")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Dataset loaded successfully!

Columns: ['State', 'District', 'Crop', 'Season', 'Irrigation_Type', 'Total_Water_Footprint(L_per_kg)', 'Category']
📦 Loading saved model from Drive...
📂 Loading pre-saved embeddings from Drive...

================= 💧 Chatbot Evaluation =================

🔹 Query 1: What is the water footprint of sugarcane in Pune district?
💬 Chatbot Reply:
 🌿 The total water footprint of **Sugarcane** in **Pune** during the **Kharif** season under **Drip** irrigation is **1870.01 L/kg**, categorized as **medium**.
------------------------------------------------------
🔹 Query 2: Which crop has the highest water footprint in Solapur?
💬 Chatbot Reply:
 💧 In **Solapur**, the crop with the **highest water footprint** is **Banana** with **4210.84 L/kg**, categorized as **high**.
------------------------------------------------------
🔹 Query 3: Compa